In [1]:
%%writefile pakage.py

import os
import tarfile
import shutil
import tensorflow as tf
import boto3

# --- Configuration ---
TAR_NAME = 'model_v2.tar.gz'
S3_BUCKET = 'sagemaker-studio-i0gutcxdy' # Replace if different
S3_PREFIX = 'frustration-model/models'

# 1. Clean up old build directories if they exist
for d in ['1', 'code']:
    if os.path.exists(d):
        shutil.rmtree(d)

os.makedirs('1', exist_ok=True)
os.makedirs('code', exist_ok=True)

# 2. Convert Keras model to SavedModel format
print("Converting autoencoder_model.keras to SavedModel format...")
try:
    model = tf.keras.models.load_model('autoencoder_model.keras')
    tf.saved_model.save(model, '1')
    print("Successfully created SavedModel bundle in '1/' directory.")
except Exception as e:
    print(f"Error converting model: {e}")

# 3. Gather inference script and Scikit-Learn artifacts into the 'code' directory
print("Staging files into 'code/' directory...")
files_to_package = [
    'inference.py',
    'requirements.txt',
    'isolation_forest.pkl',
    'scaler.pkl',
    'model_metadata.joblib'
]

for file_name in files_to_package:
    if os.path.exists(file_name):
        shutil.copy(file_name, os.path.join('code', file_name))
        print(f" -> Copied {file_name}")
    else:
        print(f" ⚠️ WARNING: {file_name} not found in the current directory!")

# 4. Create the final tarball
print(f"Packaging everything into {TAR_NAME}...")
with tarfile.open(TAR_NAME, mode='w:gz') as archive:
    archive.add('1', arcname='1')       # The deep learning weights
    archive.add('code', arcname='code') # The inference logic and sklearn artifacts

print(f"Successfully packaged {TAR_NAME}!")

# 5. (Optional) Upload directly to S3
print(f"Uploading {TAR_NAME} to s3://{S3_BUCKET}/{S3_PREFIX}/{TAR_NAME}...")
s3 = boto3.client('s3')
s3.upload_file(TAR_NAME, S3_BUCKET, f"{S3_PREFIX}/{TAR_NAME}")
print("Upload complete! You are ready to deploy.")

Writing pakage.py
